USER_STORY-1(Aniruddha Bolakhe)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Unzip the dataset
import zipfile
zip_path = '/content/drive/MyDrive/MovieLens/ml-latest-small.zip'
extract_path = '/content/drive/MyDrive/MovieLens/'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Extracted successfully!")

Extracted successfully!


In [ ]:
# Load the dataset
import pandas as pd
movies = pd.read_csv('/content/drive/MyDrive/MovieLens/ml-latest-small/movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/MovieLens/ml-latest-small/ratings.csv')
tags = pd.read_csv('/content/drive/MyDrive/MovieLens/ml-latest-small/tags.csv')

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
print("Tags shape:", tags.shape)


Movies shape: (9742, 3)
Ratings shape: (100836, 4)
Tags shape: (3683, 4)


In [ ]:
# Preview the data
print("=== MOVIES ===")
display(movies.head())
print("=== RATINGS ===")
display(ratings.head())
print("=== TAGS ===")
display(tags.head())

=== MOVIES ===


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


=== RATINGS ===


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


=== TAGS ===


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [ ]:
# Validate the data
print("Missing values in Movies:\n", movies.isnull().sum())
print("\nMissing values in Ratings:\n", ratings.isnull().sum())
print("\nMissing values in Tags:\n", tags.isnull().sum())
print("\nUnique users:", ratings['userId'].nunique())
print("Unique movies:", ratings['movieId'].nunique())
print("Total ratings:", len(ratings))

Missing values in Movies:
 movieId    0
title      0
genres     0
dtype: int64

Missing values in Ratings:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Missing values in Tags:
 userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Unique users: 610
Unique movies: 9724
Total ratings: 100836


User-Story-2(Aniruddha Bolakhe)

In [ ]:
# Step 1 - Extract year from movie title and clean title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['title_clean'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

print("Step 1 Done - Year extracted from titles")
display(movies.head())

Step 1 Done - Year extracted from titles


,movieId,title,genres,year,title_clean
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II


In [ ]:
# Step 2 - Split genres into a list
movies['genres_list'] = movies['genres'].str.split('|')

# Handle movies with no genres
movies['genres'] = movies['genres'].replace('(no genres listed)', 'Unknown')

print("Step 2 Done - Genres split into list")
display(movies[['movieId', 'title_clean', 'year', 'genres_list']].head())

Step 2 Done - Genres split into list


,movieId,title_clean,year,genres_list
0,1,Toy Story,1995,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji,1995,"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men,1995,"[Comedy, Romance]"
3,4,Waiting to Exhale,1995,"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II,1995,[Comedy]


In [ ]:
# Step 3 - Convert timestamp to readable date in ratings
import pandas as pd

ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year_rated'] = ratings['date'].dt.year
ratings['month_rated'] = ratings['date'].dt.month

# Drop raw timestamp
ratings.drop(columns=['timestamp'], inplace=True)

print("Step 3 Done - Timestamp converted to date")
display(ratings.head())

Step 3 Done - Timestamp converted to date


,userId,movieId,rating,date,year_rated,month_rated
0,1,1,4.0,2000-07-30 18:45:03,2000,7
1,1,3,4.0,2000-07-30 18:20:47,2000,7
2,1,6,4.0,2000-07-30 18:37:04,2000,7
3,1,47,5.0,2000-07-30 19:03:35,2000,7
4,1,50,5.0,2000-07-30 18:48:51,2000,7


In [ ]:
# Step 4 - Clean tags data
# Lowercase all tags
tags['tag'] = tags['tag'].str.lower().str.strip()

# Drop duplicate tags (same user tagging same movie with same tag)
tags.drop_duplicates(subset=['userId', 'movieId', 'tag'], inplace=True)

# Drop timestamp from tags
tags.drop(columns=['timestamp'], inplace=True)

print("Step 4 Done - Tags cleaned")
display(tags.head())

Step 4 Done - Tags cleaned


,userId,movieId,tag
0,2,60756,funny
1,2,60756,highly quotable
2,2,60756,will ferrell
3,2,89774,boxing story
4,2,89774,mma


In [ ]:
# Step 5 - Filter out noise
# Remove movies with less than 5 ratings
movie_rating_counts = ratings.groupby('movieId').size()
valid_movies = movie_rating_counts[movie_rating_counts >= 5].index
ratings = ratings[ratings['movieId'].isin(valid_movies)]

# Remove users with less than 5 ratings
user_rating_counts = ratings.groupby('userId').size()
valid_users = user_rating_counts[user_rating_counts >= 5].index
ratings = ratings[ratings['userId'].isin(valid_users)]

print("Step 5 Done - Noise filtered")
print("Ratings after filtering:", ratings.shape)
print("Unique users after filtering:", ratings['userId'].nunique())
print("Unique movies after filtering:", ratings['movieId'].nunique())

Step 5 Done - Noise filtered
Ratings after filtering: (90274, 6)
Unique users after filtering: 610
Unique movies after filtering: 3650


In [ ]:
# Step 6 - Save cleaned data
save_path = '/content/drive/MyDrive/MovieLens/'

movies.to_csv(save_path + 'movies_cleaned.csv', index=False)
ratings.to_csv(save_path + 'ratings_cleaned.csv', index=False)
tags.to_csv(save_path + 'tags_cleaned.csv', index=False)

print("All cleaned files saved to Google Drive ")
print("- movies_cleaned.csv")
print("- ratings_cleaned.csv")
print("- tags_cleaned.csv")

All cleaned files saved to Google Drive 
- movies_cleaned.csv
- ratings_cleaned.csv
- tags_cleaned.csv


USER-STORY-3(Aniruddha Bolakhe)

In [ ]:
# Step 1 - Movie Features: Average rating and rating count per movie
movie_features = ratings.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

# Round average rating to 2 decimal places
movie_features['avg_rating'] = movie_features['avg_rating'].round(2)

# Merge with movies dataframe
movie_features = movie_features.merge(movies[['movieId', 'title_clean', 'year', 'genres_list']], on='movieId', how='left')

print("Step 1 Done - Movie features created")
display(movie_features.head())

Step 1 Done - Movie features created


,movieId,avg_rating,rating_count,title_clean,year,genres_list
0,1,3.92,215,Toy Story,1995,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,3.43,110,Jumanji,1995,"[Adventure, Children, Fantasy]"
2,3,3.26,52,Grumpier Old Men,1995,"[Comedy, Romance]"
3,4,2.36,7,Waiting to Exhale,1995,"[Comedy, Drama, Romance]"
4,5,3.07,49,Father of the Bride Part II,1995,[Comedy]


In [ ]:
# Step 2 - User Features: Average rating given and number of movies rated
user_features = ratings.groupby('userId').agg(
    avg_rating_given=('rating', 'mean'),
    movies_rated=('movieId', 'count')
).reset_index()

user_features['avg_rating_given'] = user_features['avg_rating_given'].round(2)

print("Step 2 Done - User features created")
display(user_features.head())

Step 2 Done - User features created


,userId,avg_rating_given,movies_rated
0,1,4.36,227
1,2,3.98,27
2,3,1.48,28
3,4,3.54,199
4,5,3.64,44


In [ ]:
# Step 3 - User Favorite Genres
# Merge ratings with movie genres
ratings_with_genres = ratings.merge(movies[['movieId', 'genres_list']], on='movieId', how='left')

# Explode genres so each genre gets its own row
ratings_with_genres = ratings_with_genres.explode('genres_list')
ratings_with_genres.rename(columns={'genres_list': 'genre'}, inplace=True)

# Find top genre per user based on average rating
user_genre_preference = ratings_with_genres.groupby(['userId', 'genre'])['rating'].mean().reset_index()
user_top_genre = user_genre_preference.loc[user_genre_preference.groupby('userId')['rating'].idxmax()]
user_top_genre.rename(columns={'genre': 'top_genre', 'rating': 'top_genre_avg_rating'}, inplace=True)

# Merge into user features
user_features = user_features.merge(user_top_genre[['userId', 'top_genre']], on='userId', how='left')

print("Step 3 Done - User favorite genres extracted")
display(user_features.head())

Step 3 Done - User favorite genres extracted


,userId,avg_rating_given,movies_rated,top_genre
0,1,4.36,227,Film-Noir
1,2,3.98,27,Romance
2,3,1.48,28,Horror
3,4,3.54,199,Documentary
4,5,3.64,44,Musical


In [ ]:
# Step 4 - Build User-Item Matrix
# Rows = users, Columns = movies, Values = ratings
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

# Fill missing values with 0 (user hasn't rated that movie)
user_item_matrix.fillna(0, inplace=True)

print("Step 4 Done - User-Item Matrix created")
print("Matrix shape:", user_item_matrix.shape)
print("(Rows = Users, Columns = Movies)")
display(user_item_matrix.iloc[:5, :5])

Step 4 Done - User-Item Matrix created
Matrix shape: (610, 3650)
(Rows = Users, Columns = Movies)


movieId,1,2,3,4,5
userId,,,,,
1,4.0,0.0,4.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0


In [ ]:
# Step 5 - Save all feature files
import pickle

save_path = '/content/drive/MyDrive/MovieLens/'

# Save as CSV
movie_features.to_csv(save_path + 'movie_features.csv', index=False)
user_features.to_csv(save_path + 'user_features.csv', index=False)
user_item_matrix.to_csv(save_path + 'user_item_matrix.csv')

# Save user-item matrix as pickle too (faster to load later)
with open(save_path + 'user_item_matrix.pkl', 'wb') as f:
    pickle.dump(user_item_matrix, f)

print("All feature files saved to Google Drive ✅")
print("- movie_features.csv")
print("- user_features.csv")
print("- user_item_matrix.csv")
print("- user_item_matrix.pkl")

All feature files saved to Google Drive ✅
- movie_features.csv
- user_features.csv
- user_item_matrix.csv
- user_item_matrix.pkl


USER-STORY-4

In [ ]:
# ============================================
# US-04: Collaborative Filtering
# ============================================

import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Load data
ratings = pd.read_csv('/content/drive/MyDrive/MovieLens/ratings_cleaned.csv')
movies = pd.read_csv('/content/drive/MyDrive/MovieLens/movies_cleaned.csv')
print("Data loaded! Ratings shape:", ratings.shape)

# Build User-Item Matrix
user_item_matrix = ratings.pivot_table(
    index='userId', columns='movieId', values='rating'
).fillna(0)
print("User-Item Matrix:", user_item_matrix.shape)

# Compute User Similarity (KNN)
user_similarity_df = pd.DataFrame(
    cosine_similarity(user_item_matrix),
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
print("User Similarity Matrix computed!")

# SVD - Matrix Factorization
U, sigma, Vt = np.linalg.svd(user_item_matrix.values, full_matrices=False)
k = 50
predicted_matrix = np.dot(np.dot(U[:, :k], np.diag(sigma[:k])), Vt[:k, :])
predicted_df = pd.DataFrame(
    predicted_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)
print("SVD Matrix Factorization done!")

# Train/Test Split
train, test = train_test_split(ratings, test_size=0.2, random_state=42)
test_sample = test.sample(500, random_state=42).copy()
print(f"Train: {len(train)} | Test: {len(test)}")

# Evaluate KNN
def predict_knn(user_id, movie_id):
    if movie_id not in user_item_matrix.columns:
        return ratings['rating'].mean()
    sim_users = user_similarity_df[user_id].drop(user_id).nlargest(20).index
    sim_ratings = user_item_matrix.loc[sim_users, movie_id]
    sim_ratings = sim_ratings[sim_ratings > 0]
    return sim_ratings.mean() if len(sim_ratings) > 0 else ratings['rating'].mean()

test_sample['knn_pred'] = test_sample.apply(
    lambda r: predict_knn(r['userId'], r['movieId']), axis=1
)
knn_rmse = np.sqrt(mean_squared_error(test_sample['rating'], test_sample['knn_pred']))

# Evaluate SVD
def predict_svd(user_id, movie_id):
    if user_id not in predicted_df.index or movie_id not in predicted_df.columns:
        return ratings['rating'].mean()
    return predicted_df.loc[user_id, movie_id]

test_sample['svd_pred'] = test_sample.apply(
    lambda r: predict_svd(r['userId'], r['movieId']), axis=1
)
svd_rmse = np.sqrt(mean_squared_error(test_sample['rating'], test_sample['svd_pred']))

# Compare Models
print("\n=== MODEL COMPARISON ===")
print(f"KNN RMSE: {knn_rmse:.4f}")
print(f"SVD RMSE: {svd_rmse:.4f}")
best_model = 'svd' if svd_rmse < knn_rmse else 'knn'
print(f"✅ Best Model: {best_model.upper()}")

# Generate Top 10 Recommendations for User 1
def get_recommendations(user_id, top_n=10):
    rated = set(ratings[ratings['userId'] == user_id]['movieId'])
    unrated = set(user_item_matrix.columns) - rated

    if best_model == 'svd':
        preds = predicted_df.loc[user_id, list(unrated)]
    else:
        sim_users = user_similarity_df[user_id].drop(user_id).nlargest(20).index
        preds = ratings[
            ratings['userId'].isin(sim_users) & ratings['movieId'].isin(unrated)
        ].groupby('movieId')['rating'].mean()

    top_ids = preds.nlargest(top_n).index.tolist()
    result = movies[movies['movieId'].isin(top_ids)][['movieId', 'title_clean', 'genres']].copy()
    result['predicted_rating'] = result['movieId'].map(dict(zip(top_ids, preds[top_ids].round(2))
    ))
    return result.sort_values('predicted_rating', ascending=False)

print("\n=== TOP 10 RECOMMENDATIONS FOR USER 1 ===")
display(get_recommendations(1))

# Save Model
save_path = '/content/drive/MyDrive/MovieLens/'
model_data = {
    'best_model': best_model,
    'user_item_matrix': user_item_matrix,
    'user_similarity': user_similarity_df,
    'predicted_matrix': predicted_df,
    'ratings': ratings,
    'movies': movies
}
with open(save_path + 'collaborative_filtering_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("\nModel saved ✅ - collaborative_filtering_model.pkl")

Data loaded! Ratings shape: (90274, 6)
User-Item Matrix: (610, 3650)
User Similarity Matrix computed!
SVD Matrix Factorization done!
Train: 72219 | Test: 18055

=== MODEL COMPARISON ===
KNN RMSE: 1.0004
SVD RMSE: 2.0914
✅ Best Model: KNN

=== TOP 10 RECOMMENDATIONS FOR USER 1 ===


,movieId,title_clean,genres,predicted_rating
198,232,Eat Drink Man Woman (Yin shi nan nu),Comedy|Drama|Romance,5.0
878,1172,Cinema Paradiso (Nuovo cinema Paradiso),Drama,5.0
883,1178,Paths of Glory,Drama|War,5.0
994,1296,"Room with a View, A",Drama|Romance,5.0
1532,2067,Doctor Zhivago,Drama|Romance|War,5.0
1568,2106,Swing Kids,Drama|War,5.0
1665,2240,My Bodyguard,Drama,5.0
1703,2290,Stardust Memories,Comedy|Drama,5.0
2030,2702,Summer of Sam,Drama,5.0
2276,3019,Drugstore Cowboy,Crime|Drama,5.0



Model saved ✅ - collaborative_filtering_model.pkl


 USER STORY- 5(VIDUSHI NEGI)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# STEP 1: Create genres_list
movies['genres_list'] = movies['genres'].apply(lambda x: x.split('|'))

# STEP 2: Convert to string
movies['genres_str'] = movies['genres_list'].apply(lambda x: " ".join(x))

# STEP 3: TF-IDF
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_str'])

# STEP 4: Similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# STEP 5: Recommendation function
def content_recommend(movie_name, top_n=10):
    match = movies[movies['title_clean'].str.lower() == movie_name.lower()]
    if match.empty:
        match = movies[movies['title'].str.lower() == movie_name.lower()]
    if match.empty:
        return f"Movie '{movie_name}' not found"

    idx = match.index[0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in scores]
    return movies.iloc[movie_indices][['title_clean', 'genres']]

# TEST
display(content_recommend("Toy Story (1995)"))

# SAVE
save_path = '/content/drive/MyDrive/MovieLens/'
with open(save_path + 'content_based_model.pkl', 'wb') as f:
    pickle.dump({'cosine_sim': cosine_sim, 'movies': movies}, f)

print("Content-Based model saved ✅ - content_based_model.pkl")

,title_clean,genres
1706,Antz,Adventure|Animation|Children|Comedy|Fantasy
2355,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The",Adventure|Animation|Children|Comedy|Fantasy
3000,"Emperor's New Groove, The",Adventure|Animation|Children|Comedy|Fantasy
3568,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third,Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The",Adventure|Animation|Children|Comedy|Fantasy
7760,Asterix and the Vikings (Astérix et les Vikings),Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo,Adventure|Animation|Children|Comedy|Fantasy


Content-Based model saved ✅ - content_based_model.pkl


USER STORY 6-(VIDUSHI NEGI)

In [ ]:
# STEP 1: CREATE USER-ITEM MATRIX
user_item_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

print("Matrix shape:", user_item_matrix.shape)

Matrix shape: (610, 3650)


In [ ]:
# STEP 2: APPLY SVD
import numpy as np
from scipy.sparse.linalg import svds

matrix = user_item_matrix.values

# Apply SVD
U, sigma, Vt = svds(matrix, k=50)

# Convert sigma to diagonal matrix
sigma = np.diag(sigma)

# Reconstruct predicted ratings
predicted_ratings = np.dot(np.dot(U, sigma), Vt)

print("SVD Model Built Successfully!")

SVD Model Built Successfully!


In [ ]:
# STEP 3: CREATE RECOMMENDATION FUNCTION
def recommend_movies_svd(user_id, top_n=10):
    user_index = user_item_matrix.index.get_loc(user_id)
    user_ratings = predicted_ratings[user_index]

    # Sort movies by predicted rating
    top_movies = np.argsort(user_ratings)[::-1][:top_n]

    return user_item_matrix.columns[top_movies]

In [ ]:
# STEP 4: CONVERT TO MOVIE NAMES
def recommend_movies_names(user_id, top_n=10):
    movie_ids = recommend_movies_svd(user_id, top_n)
    return movies[movies['movieId'].isin(movie_ids)][['title_clean', 'genres']]

In [ ]:
# TEST
print("=== TOP 10 RECOMMENDATIONS FOR USER 1 ===")
display(recommend_movies_names(1))

=== TOP 10 RECOMMENDATIONS FOR USER 1 ===


,title_clean,genres
43,Seven (a.k.a. Se7en),Mystery|Thriller
224,Star Wars: Episode IV - A New Hope,Action|Adventure|Sci-Fi
520,Fargo,Comedy|Crime|Drama|Thriller
898,Star Wars: Episode V - The Empire Strikes Back,Action|Adventure|Sci-Fi
900,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
911,Star Wars: Episode VI - Return of the Jedi,Action|Adventure|Sci-Fi
990,Indiana Jones and the Last Crusade,Action|Adventure
1503,Saving Private Ryan,Action|Drama|War
1734,American History X,Crime|Drama
2145,American Beauty,Drama|Romance


In [ ]:
# ============================================
# SAVE SINGLE COMBINED MODEL PKL
# ============================================
import pickle
import numpy as np

# Compare RMSE of both models and pick best
# KNN RMSE from US-04
knn_rmse = np.sqrt(mean_squared_error(test_sample['rating'], test_sample['knn_pred']))
# SVD RMSE from US-04
svd_rmse = np.sqrt(mean_squared_error(test_sample['rating'], test_sample['svd_pred']))

best_model = 'svd' if svd_rmse < knn_rmse else 'knn'
print(f"Best Model Selected: {best_model.upper()}")
print(f"KNN RMSE: {knn_rmse:.4f}")
print(f"SVD RMSE: {svd_rmse:.4f}")

# Combine everything into one pkl
combined_model = {
    # Best model info
    'best_model': best_model,

    # Collaborative Filtering data
    'user_item_matrix': user_item_matrix,
    'user_similarity': user_similarity_df,
    'predicted_matrix': predicted_df,

    # Matrix Factorization data
    'predicted_ratings': predicted_ratings,

    # Content Based data
    'cosine_sim': cosine_sim,

    # Data
    'ratings': ratings,
    'movies': movies
}

save_path = '/content/drive/MyDrive/MovieLens/'
with open(save_path + 'final_model.pkl', 'wb') as f:
    pickle.dump(combined_model, f)

print("\nSingle combined model saved ✅ - final_model.pkl")
print(f"Best model inside: {best_model.upper()}")

Best Model Selected: KNN
KNN RMSE: 1.0004
SVD RMSE: 2.0914


OSError: [Errno 107] Transport endpoint is not connected

KHUSHI -USER STORY_09 NCF

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

ratings = pd.read_csv('/content/drive/MyDrive/MovieLens/ratings_cleaned.csv')

user_enc = LabelEncoder()
ratings['user'] = user_enc.fit_transform(ratings['userId'].values)

item_enc = LabelEncoder()
ratings['movie'] = item_enc.fit_transform(ratings['movieId'].values)

n_users = ratings['user'].nunique()
n_movies = ratings['movie'].nunique()

X = ratings[['user', 'movie']].values
y = ratings['rating'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Setup Complete!")
print(f"We have {n_users} users and {n_movies} movies ready for the AI brain.")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout


user_input = Input(shape=[1], name="User-Input")
movie_input = Input(shape=[1], name="Movie-Input")

user_embedding = Embedding(n_users, 50, name="User-Embedding")(user_input)
movie_embedding = Embedding(n_movies, 50, name="Movie-Embedding")(movie_input)


user_vec = Flatten(name="Flatten-Users")(user_embedding)
movie_vec = Flatten(name="Flatten-Movies")(movie_embedding)


concat = Concatenate()([user_vec, movie_vec])


fc1 = Dense(128, activation='relu')(concat)
fc1 = Dropout(0.2)(fc1)
fc2 = Dense(64, activation='relu')(fc1)
fc2 = Dropout(0.2)(fc2)

out = Dense(1)(fc2)

model = Model([user_input, movie_input], out)
model.compile(optimizer='adam', loss='mean_squared_error')

model.summary()

In [ ]:

# epochs=10
# batch_size=64
history = model.fit(
    [X_train[:, 0], X_train[:, 1]],
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=([X_test[:, 0], X_test[:, 1]], y_test),
    verbose=1
)


print("\nTraining Complete!")

In [ ]:
from sklearn.metrics import mean_squared_error


y_pred = model.predict([X_test[:, 0], X_test[:, 1]])


rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"=== MODEL ACCURACY ===")
print(f"NCF Model RMSE: {rmse:.4f}")



In [ ]:
def get_precision_recall_at_k(predictions, k=10, threshold=3.5):
    # 1. Group predictions by UserID
    user_est_true = {}
    for user_id, movie_id, actual, est in predictions:
        if user_id not in user_est_true:
            user_est_true[user_id] = []
        user_est_true[user_id].append((est, actual))

    precisions = {}
    recalls = {}

    for user_id, user_ratings in user_est_true.items():

        user_ratings.sort(key=lambda x: x[0], reverse=True)

        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])


        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        # Precision@K
        precisions[user_id] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

        # Recall@K
        recalls[user_id] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return np.mean(list(precisions.values())), np.mean(list(recalls.values()))


test_results = []
for i in range(len(X_test)):
    test_results.append((X_test[i, 0], X_test[i, 1], y_test[i], y_pred[i][0]))

p_at_k, r_at_k = get_precision_recall_at_k(test_results, k=10)

print(f"Precision@10: {p_at_k:.4f}")
print(f"Recall@10: {r_at_k:.4f}")

In [ ]:
import pickle


model_save_path = '/content/drive/MyDrive/MovieLens/ncf_model.h5'
encoder_save_path = '/content/drive/MyDrive/MovieLens/user_item_encoders.pkl'


model.save(model_save_path)


encoders = {
    'user_encoder': user_enc,
    'item_encoder': item_enc
}

with open(encoder_save_path, 'wb') as f:
    pickle.dump(encoders, f)

print("✅ Success! Model and Encoders are saved in your Google Drive.")
print(f"Path 1: {model_save_path}")
print(f"Path 2: {encoder_save_path}")